In [1]:
import pandas as pd
import yfinance as yf
import itertools
import talib
from backtesting import Backtest, Strategy



In [2]:
"""
Параметры:
ticker (str): Тикер актива (напр, 'BTC-USD', 'AAPL').
start_date (str): Начальная дата интервала, в формате 'YYYY-MM-DD'.
end_date (str): Конечная дата интервала, в формате 'YYYY-MM-DD'.
interval (str): Интервал данных, одно из значений '1d', '1h', '4h', or '5m'.

Возвращаемое значение - pd.DataFrame с данными
"""

def get_data(ticker, start_date, end_date, interval='1d'):
    df = yf.download(ticker, start=start_date, end=end_date, interval=interval)
    df = df.reset_index()
    return df


In [3]:
"""
Класс для тестирования стратегий для backtesting, 
основан на Signal в данных
"""
class TestStrategyClass(Strategy):

    def init(self):
        self.signal = self.I(lambda: self.data.Signal)
        self.previous_signal = 0

    def next(self):
        current_signal = self.signal[-1]

        if current_signal != self.previous_signal:
            if current_signal == 1:
                if self.position.is_short:
                    self.position.close()
                    
                if not self.position.is_long:
                    self.buy()
                    
            elif current_signal == -1:
                if self.position.is_long:
                    self.position.close()
                   
                if not self.position.is_short:
                    self.sell()
                    
            elif current_signal == 0:
                if self.position:
                    self.position.close()
                    

        self.previous_signal = current_signal

In [4]:
"""
    Первая - самая простая стратегия на индикаторах - пересечение стользящих средних 
    Параметры первого и второго периода скользящих средних передаются в параметре params
    На выходе - DataFrame с рассчитанными сигналом.

    Параметры стратегии, которые будем подбирать:
    SMA_short_period
    SMA_long_period
"""
def calc_strategy_sma(data, params):
    df = data.copy()

    short_period = params['SMA_short_period']
    long_period = params['SMA_long_period']

    df['SMA_SHORT'] = talib.SMA(df["Close"], timeperiod=short_period)
    df['SMA_LONG'] = talib.SMA(df["Close"], timeperiod=long_period)

    df['Signal'] = 0  
    df.loc[df['SMA_SHORT'] > df['SMA_LONG'], 'Signal'] = 1 # Сигнал на покупку
    df.loc[df['SMA_SHORT'] < df['SMA_LONG'], 'Signal'] = -1 # Сигнал на продажу
    
    return df[["Date", "Open", "High", "Low", "Close", "Volume","Signal"]]

In [5]:
"""
    Более сложная стратегия на индикаторах. (мультииндикаторная стратегия)
    Суть в том, чтобы использовать разные типы индикаторов, которые будут отражать информацию разных типов.
    RSI - индикатор импульса, позволяет определить направление и силу текущего ценового тренда
    OBV - индикатор объема, увеличение объема торгов часто сигнализирует о смене тренда
    Полосы Боллинжера - индикатор следования за трендом

    Сама стратегия:
    Сигнал на покупку:
        1. Закрытие цены выше средней полосы Боллинджера
        2. RSI выше уровня 50
        3. OBV растет

    Сигнал на продажу:
        1. Закрытие цены ниже средней полосы Боллинджера
        2. RSI ниже уровня 50
        3. OBV падает

    Параметры стратегии, которые будем подбирать:
    RSI_Period
    Bollinger_Period

"""
def calc_strategy_indicators(data, params):
    df = data.copy()
    
    RSI_Period = params['RSI_Period']
    Bollinger_Period = params['Bollinger_Period']
   
    df['RSI'] = talib.RSI(df['Close'], timeperiod=RSI_Period)
    df['OBV'] = talib.OBV(df['Close'], df['Volume'])
    df['Bollinger_High'], df['Bollinger_Middle'], df['Bollinger_Low'] = talib.BBANDS(df['Close'], timeperiod=Bollinger_Period, nbdevup=2, nbdevdn=2, matype=0)
    df = df.dropna()
    
    df['Signal_RSI'] = 0
    df['Signal_OBV'] = 0
    df['Signal_Bollnger'] = 0
    df['Signal'] = 0
    
    df.loc[df['RSI'] < 50, 'Signal_RSI'] = 1 
    df.loc[df['RSI'] > 50, 'Signal_RSI'] = -1  
    
    df['OBV_Diff'] = df['OBV'].diff()
    df.loc[df['OBV_Diff'] > 0, 'Signal_OBV'] = 1  
    df.loc[df['OBV_Diff'] < 0, 'Signal_OBV'] = -1  
    
    df.loc[df['Close'] > df['Bollinger_Middle'], 'Signal_Bollnger'] = 1 
    df.loc[df['Close'] < df['Bollinger_Middle'], 'Signal_Bollnger'] = -1 
 
     # Объединяем сигналы
    df.loc[(df['Signal_RSI']==1) & (df['Signal_OBV']==1) & (df['Signal_Bollnger']==1), 'Signal'] = 1  # Buy signal
    df.loc[(df['Signal_RSI']==-1) & (df['Signal_OBV']==-1) & (df['Signal_Bollnger']==-1), 'Signal'] = -1  # Sell signal


    return df  


In [6]:
def backtest_strategy(df, strategy_class, strategy_function, params, plot=False):
    """
    Запускает бэктест с переданными параметрами стратегии.

    :param df: DataFrame с данными для бэктеста.
    :param strategy_class: Класс стратегии для бэктеста.
    :param params: Словарь с параметрами стратегии.
    :return: Статистика бэктеста.
    """
    # Применяем стратегию с переданными параметрами
    df = strategy_function(df.copy(), params)

    # Подготовка данных для бэктеста
    bt_df = df.copy()
    #bt_df.columns = bt_df.columns.str.capitalize()
    bt_df.rename(columns={'Date': 'Datetime'}, inplace=True)
    bt_df["Datetime"] = pd.to_datetime(bt_df["Datetime"])
    bt_df.set_index('Datetime', inplace=True)
   
    # Создаем объект класса Backtest с текущей стратегией
    bt = Backtest(bt_df, strategy_class, cash=500000, commission=.002, exclusive_orders=True, margin=0.1)

    # Запускаем бэктест
    stats = bt.run()
    if plot:
        bt.plot(
        plot_equity=True,
        plot_drawdown=True,
        relative_equity=False,
        )
    return stats

In [7]:

def get_best_strategy_sma(buffer, strategy_class):
    # Задаем возможные значения для параметров стратегии
    short_period_period_list = [9, 10, 7 ]
    long_period_period_list = [26, 15, 14 ]
 
    # Для хранения лучших параметров и лучшего результата
    best_params = None
    best_performance = -float('inf') 

    # Проходим по всем комбинациям параметров
    for short_period, long_period in itertools.product(short_period_period_list, long_period_period_list):
        
        # Создаем словарь с текущими параметрами
        params = {
            'SMA_short_period': short_period,
            'SMA_long_period': long_period,
        }
        best_params = params
        # Запускаем бэктест с текущими параметрами
        stats = backtest_strategy(buffer.copy(), strategy_class, calc_strategy_sma, params)

        # Определяем метрику, по которой будем выбирать лучшую стратегию (например, по профит фактору)
        performance = stats['Profit Factor']  
        
        print(f"Params: {params}")
        print(f"Performance I: {performance}")

        # Сравниваем с лучшим результатом и сохраняем лучшие параметры
        if performance > best_performance:
            best_performance = performance
            best_params = params

    print(f"Best Performance: {best_performance}")
    print(f"Best Parameters: {best_params}")
    return best_params

In [8]:
def get_best_strategy_indicators(buffer, strategy_class):
    # Задаем возможные значения для параметров стратегии
    rsi_period_list = [14, 7, 7]
    bollinger_period_list = [14, 14, 10]
 
    # Для хранения лучших параметров и лучшего результата
    best_params = None
    best_performance = -float('inf') 

     # Проходим по всем комбинациям параметров
    for rsi_period, bollinger_period in itertools.product(rsi_period_list, bollinger_period_list):
        
        # Создаем словарь с текущими параметрами
        params = {
            'RSI_Period': rsi_period,
            'Bollinger_Period': bollinger_period,
        }
        best_params = params
        # Запускаем бэктест с текущими параметрами
        stats = backtest_strategy(buffer.copy(), strategy_class, calc_strategy_indicators, params)
        # print(stats[:27])


        # Определяем метрику, по которой будем выбирать лучшую стратегию (например, по профит фактору)
        performance = stats['Profit Factor']  

        # Сравниваем с лучшим результатом и сохраняем лучшие параметры
        if performance > best_performance:
            best_performance = performance
            best_params = params

    print(f"Best Performance: {best_performance}")
    print(f"Best Parameters: {best_params}")
    return best_params

In [9]:
# Загружаем и подготавливаем данные
start_date = '2022-09-01'
end_date = '2023-06-01'
interval = '1d'  # or '1h', '4h', '5m'

df = get_data('BTC-USD', start_date, end_date, interval)
df = df.dropna()

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

df.tail(10)


[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
263,2023-05-22,26851.277344,27045.734375,26549.734375,26749.892578,11056770492
264,2023-05-23,27225.726562,27434.683594,26816.179688,26855.960938,13697203143
265,2023-05-24,26334.818359,27224.603516,26106.576172,27224.603516,16299104428
266,2023-05-25,26476.207031,26591.519531,25890.593750,26329.460938,13851122697
267,2023-05-26,26719.291016,26916.669922,26343.949219,26474.181641,12711619225
268,2023-05-27,26868.353516,26888.882812,26621.140625,26720.181641,7892015141
269,2023-05-28,28085.646484,28193.449219,26802.751953,26871.158203,14545229578
270,2023-05-29,27745.884766,28432.039062,27563.876953,28075.591797,15181308984
271,2023-05-30,27702.349609,28044.759766,27588.501953,27745.123047,13251081851
272,2023-05-31,27219.658203,27831.677734,26866.453125,27700.529297,15656371534


In [10]:
# Поиск оптимальных параметров для стратегии пересечения SMA
train_size = 90  # Размер окна тренировки
test_size = 60    # Размер тестового окна

# Инициализация DataFrame для сигналов
signals_df = pd.DataFrame()

# Определение количества итераций
num_iterations = (len(df) - train_size) // test_size

for i in range(num_iterations + 1):
    # Определение границ обучающего и тестового окон
    start_train = i * test_size
    end_train = start_train + train_size
    start_test = end_train
    end_test = start_test + test_size
    
    # Если конец тестового окна выходит за пределы данных, обрезаем его
    if end_test > len(df):
        end_test = len(df)
    
    # Определяем окна для тренировки и тестирования
    train_data = df.iloc[start_train:end_train].copy()
    test_data = df.iloc[start_test:end_test].copy()
    
    
    # Оптимизация на тренировочном окне
    best_params = get_best_strategy_sma(train_data, TestStrategyClass)
    
    # Объединяем данные тренировки и теста
    combined_data = pd.concat([train_data, test_data]).reset_index(drop=True)
    
    # Применяем стратегию на объединенном окне с оптимальными параметрами
    combined_with_signal = calc_strategy_sma(combined_data.copy(), best_params)
    
    # Извлекаем только часть данных, относящуюся к тестовому окну
    test_with_signal = combined_with_signal.iloc[-test_size:].copy()
    
    # Добавляем сигналы из тестового окна в signals_df
    signals_df = pd.concat([signals_df, test_with_signal], ignore_index=True)



Params: {'SMA_short_period': 9, 'SMA_long_period': 26}
Performance I: 0.0
Params: {'SMA_short_period': 9, 'SMA_long_period': 15}
Performance I: 1.1709669860714893
Params: {'SMA_short_period': 9, 'SMA_long_period': 14}
Performance I: 0.7897235057245843
Params: {'SMA_short_period': 10, 'SMA_long_period': 26}
Performance I: 0.0
Params: {'SMA_short_period': 10, 'SMA_long_period': 15}
Performance I: 0.77322430167443
Params: {'SMA_short_period': 10, 'SMA_long_period': 14}
Performance I: 0.8765417755606563
Params: {'SMA_short_period': 7, 'SMA_long_period': 26}
Performance I: 0.0
Params: {'SMA_short_period': 7, 'SMA_long_period': 15}
Performance I: 2.9568500087953815
Params: {'SMA_short_period': 7, 'SMA_long_period': 14}
Performance I: 1.2153356457885813
Best Performance: 2.9568500087953815
Best Parameters: {'SMA_short_period': 7, 'SMA_long_period': 14}
Params: {'SMA_short_period': 9, 'SMA_long_period': 26}
Performance I: 4.147953944600946
Params: {'SMA_short_period': 9, 'SMA_long_period': 15}

In [11]:
s_df = signals_df.copy().reset_index(drop=True)

s_df.rename(columns={'Date': 'Datetime'}, inplace=True)
s_df["Datetime"] = pd.to_datetime(s_df["Datetime"])
s_df.set_index('Datetime', inplace=True)
s_df = s_df.sort_index()
s_df


st = Backtest(s_df, TestStrategyClass, cash=500000, commission=0.002, exclusive_orders=True, margin=0.1)

# Запускаем бэктест
stats = st.run()
print(stats)
st.plot(
plot_equity=True,
plot_drawdown=True,
relative_equity=False,
)


Start                     2022-11-30 00:00:00
End                       2023-05-31 00:00:00
Duration                    182 days 00:00:00
Exposure Time [%]                   99.166667
Equity Final [$]                  55813.87973
Equity Peak [$]                1469790.118094
Return [%]                         -88.837224
Buy & Hold Return [%]               58.543571
Return (Ann.) [%]                  -98.738905
Volatility (Ann.) [%]             1758.873928
Sharpe Ratio                              0.0
Sortino Ratio                             0.0
Calmar Ratio                              0.0
Max. Drawdown [%]                  -97.994752
Avg. Drawdown [%]                  -24.415108
Max. Drawdown Duration      110 days 00:00:00
Avg. Drawdown Duration       20 days 00:00:00
# Trades                                   15
Win Rate [%]                             20.0
Best Trade [%]                      35.090883
Worst Trade [%]                     -7.268885
Avg. Trade [%]                    

/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest valu

GridPlot(id='p1402', ...)

In [12]:
train_size = 90  # Размер окна тренировки
test_size = 30    # Размер тестового окна

# Инициализация DataFrame для сигналов
signals_df = pd.DataFrame()

# Определение количества итераций
num_iterations = (len(df) - train_size) // test_size

for i in range(num_iterations + 1):
    # Определение границ обучающего и тестового окон
    start_train = i * test_size
    end_train = start_train + train_size
    start_test = end_train
    end_test = start_test + test_size
    
    # Если конец тестового окна выходит за пределы данных, обрезаем его
    if end_test > len(df):
        end_test = len(df)
    
    # Определяем окна для тренировки и тестирования
    train_data = df.iloc[start_train:end_train].copy()
    test_data = df.iloc[start_test:end_test].copy()
    
    
    # Оптимизация на тренировочном окне
    best_params = get_best_strategy_indicators(train_data, TestStrategyClass)
    
    # Объединяем данные тренировки и теста
    combined_data = pd.concat([train_data, test_data]).reset_index(drop=True)
    
    # Применяем стратегию на объединенном окне с оптимальными параметрами
    combined_with_signal = calc_strategy_indicators(combined_data.copy(), best_params)
    
    # Извлекаем только часть данных, относящуюся к тестовому окну
    test_with_signal = combined_with_signal.iloc[-test_size:].copy()
    
    # Добавляем сигналы из тестового окна в signals_df
    signals_df = pd.concat([signals_df, test_with_signal], ignore_index=True)

kf.service.services: KApplicationTrader: mimeType "x-scheme-handler/file" not found
Found ffmpeg: /opt/yandex/browser-beta/libffmpeg.so
	avcodec: 4002148
	avformat: 3999332
	avutil: 3876196
FFmpeg version is too new. Need:
	avcodec: 3999076
	avformat: 3998564
	avutil: 3871588


find_ffmpeg failed, using the integrated library.
Opening in existing browser session.
Best Performance: 10.398439053495395
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 28.748472601295095
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 8.600451958701841
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 1.6841560404880553
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 5.620448220020175
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 2.905178523262684
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}
Best Performance: 0.674285296000598
Best Parameters: {'RSI_Period': 7, 'Bollinger_Period': 10}


In [13]:
s_df = signals_df.copy()

s_df.rename(columns={'Date': 'Datetime'}, inplace=True)
s_df["Datetime"] = pd.to_datetime(s_df["Datetime"])
s_df.set_index('Datetime', inplace=True)
s_df = s_df.sort_index()
s_df


st = Backtest(s_df, TestStrategyClass, cash=500000, commission=0.002, exclusive_orders=True, margin=0.1)

# Запускаем бэктест
stats = st.run()
print(stats[:27])
st.plot(
plot_equity=True,
plot_drawdown=True,
relative_equity=False,
)

Start                     2022-11-30 00:00:00
End                       2023-05-31 00:00:00
Duration                    182 days 00:00:00
Exposure Time [%]                   15.714286
Equity Final [$]               1049126.112738
Equity Peak [$]                1508202.683051
Return [%]                         109.825223
Buy & Hold Return [%]               58.543571
Return (Ann.) [%]                  338.486876
Volatility (Ann.) [%]              438.889786
Sharpe Ratio                         0.771234
Sortino Ratio                        10.49734
Calmar Ratio                        11.120298
Max. Drawdown [%]                  -30.438652
Avg. Drawdown [%]                   -20.78801
Max. Drawdown Duration       62 days 00:00:00
Avg. Drawdown Duration       20 days 00:00:00
# Trades                                   13
Win Rate [%]                        69.230769
Best Trade [%]                       3.495456
Worst Trade [%]                     -1.550283
Avg. Trade [%]                    

/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/home/olga/Projects/ml/homework-test/hw2/.venv/lib/python3.9/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest valu

GridPlot(id='p1834', ...)

kf.service.services: KApplicationTrader: mimeType "x-scheme-handler/file" not found
Found ffmpeg: /opt/yandex/browser-beta/libffmpeg.so
	avcodec: 4002148
	avformat: 3999332
	avutil: 3876196
FFmpeg version is too new. Need:
	avcodec: 3999076
	avformat: 3998564
	avutil: 3871588


find_ffmpeg failed, using the integrated library.
Opening in existing browser session.
